> Repository edition: run from the repository root using an environment with the dependencies in `requirements.txt`. Machine-specific environment paths have been replaced. Historical outputs are retained except environment checks; they are not evidence of a fresh rerun. See `docs/REPRODUCIBILITY.md`.


In [1]:
import sys
print(sys.executable)

c:\Users\AG\BeatrizGIT\Master\.venv\Scripts\python.exe


In [2]:
from pathlib import Path
import numpy as np

# Check the working directory and required files
print("Working directory:", Path.cwd())
print("Initial data folder found:", Path("initial_data").is_dir())
print("Week 1 inputs found:", Path("week_1/inputs.txt").is_file())
print("Week 1 outputs found:", Path("week_1/outputs.txt").is_file())


Working directory: c:\Users\AG\BeatrizGIT\Master\CapstoneProject
Initial data folder found: True
Week 1 inputs found: True
Week 1 outputs found: True


In [3]:
import ast
import re

# Read the Week 1 feedback files
inputs_text = Path("week_1/inputs.txt").read_text(encoding="utf-8-sig")
outputs_text = Path("week_1/outputs.txt").read_text(encoding="utf-8-sig")

# Convert array([...]) into plain lists, without executing file contents
inputs_text = re.sub(r"array\((\[[^\[\]]*\])\)", r"\1", inputs_text)

week1_inputs = ast.literal_eval(inputs_text)
week1_outputs = ast.literal_eval(outputs_text)

assert len(week1_inputs) == len(week1_outputs) == 8

# Store the updated data for each function
updated_data = {}

for function_id in range(1, 9):
    folder = Path("initial_data") / f"function_{function_id}"

    # Reload the original data on every execution
    X_initial = np.load(folder / "initial_inputs.npy", allow_pickle=False)
    y_initial = np.load(
        folder / "initial_outputs.npy", allow_pickle=False
    ).reshape(-1)

    # Read the corresponding input-output pair from Week 1
    new_input = np.asarray(week1_inputs[function_id - 1], dtype=float)
    new_output = float(week1_outputs[function_id - 1])

    assert new_input.shape == (X_initial.shape[1],)
    assert np.isfinite(new_input).all() and np.isfinite(new_output)

    # Add one observation to the initial data
    X_updated = np.vstack((X_initial, new_input))
    y_updated = np.append(y_initial, new_output)

    updated_data[function_id] = {
        "X": X_updated,
        "y": y_updated
    }

    print(
        f"Function {function_id}: "
        f"{len(y_initial)} → {len(y_updated)} observations | "
        f"{X_updated.shape[1]} variables"
    )

Function 1: 10 → 11 observations | 2 variables
Function 2: 10 → 11 observations | 2 variables
Function 3: 15 → 16 observations | 3 variables
Function 4: 30 → 31 observations | 4 variables
Function 5: 20 → 21 observations | 4 variables
Function 6: 20 → 21 observations | 5 variables
Function 7: 30 → 31 observations | 6 variables
Function 8: 40 → 41 observations | 8 variables


In [4]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel

# Use the updated Function 1 data
X_1 = updated_data[1]["X"]
y_1 = updated_data[1]["y"]

models_1 = {}

# Compare the original bounds with wider bounds
for name, upper_bound in [("A_original", 2.0), ("B_wider", 10.0)]:

    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(
            length_scale=[0.2] * X_1.shape[1],
            length_scale_bounds=(0.02, upper_bound),
            nu=2.5
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1)
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    model.fit(X_1, y_1)
    models_1[name] = model

    print(f"\nModel: {name}")
    print("Fitted kernel:", model.kernel_)
    print(
        "Log-marginal likelihood:",
        model.log_marginal_likelihood_value_
    )
    

c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(



Model: A_original
Fitted kernel: 1.02**2 * Matern(length_scale=[2, 0.0267], nu=2.5) + WhiteKernel(noise_level=2.86e-08)
Log-marginal likelihood: -15.305280545956165

Model: B_wider
Fitted kernel: 1.02**2 * Matern(length_scale=[10, 0.0269], nu=2.5) + WhiteKernel(noise_level=1.6e-08)
Log-marginal likelihood: -15.297823164356938


c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [5]:
# Generate the same candidate points for both models
rng_1 = np.random.default_rng(42)
candidates_1 = rng_1.uniform(0, 0.999999, size=(100000, 2))

# Round first so predictions correspond to portal-ready coordinates
candidates_1 = np.unique(np.round(candidates_1, 6), axis=0)

# Calculate the distance to the nearest existing observation
nearest_distance_1 = np.min(
    np.linalg.norm(
        candidates_1[:, None, :] - X_1[None, :, :],
        axis=2
    ),
    axis=1
)

# Exclude points that have already been evaluated
keep = nearest_distance_1 > 1e-6
candidates_1 = candidates_1[keep]
nearest_distance_1 = nearest_distance_1[keep]

# Compare UCB proposals without selecting a final model yet
beta_1 = 2.0
proposals_1 = {}

for name, model in models_1.items():
    mean, std = model.predict(candidates_1, return_std=True)
    ucb = mean + beta_1 * std

    index = np.argmax(ucb)
    query = candidates_1[index]

    proposals_1[name] = {
        "query": query,
        "mean": float(mean[index]),
        "std": float(std[index]),
        "ucb": float(ucb[index])
    }

    print(f"\nModel: {name}")
    print("Proposed query:", "-".join(f"{x:.6f}" for x in query))
    print(f"Predicted output: {mean[index]:.6e}")
    print(f"Predictive standard deviation: {std[index]:.6e}")
    print(f"UCB score: {ucb[index]:.6e}")
    print(f"Distance to nearest observation: {nearest_distance_1[index]:.6f}")


Model: A_original
Proposed query: 0.384552-0.349652
Predicted output: -2.779497e-04
Predictive standard deviation: 1.043355e-03
UCB score: 1.808760e-03
Distance to nearest observation: 0.203749

Model: B_wider
Proposed query: 0.329776-0.348685
Predicted output: -2.782960e-04
Predictive standard deviation: 1.044499e-03
UCB score: 1.810702e-03
Distance to nearest observation: 0.216702


In [6]:
# Store the selected model and query for Function 1, round 2
selected_model_1 = models_1["A_original"]
query_1_round2 = proposals_1["A_original"]["query"].copy()

print("Function 1 — round 2:")
print("-".join(f"{x:.6f}" for x in query_1_round2))

Function 1 — round 2:
0.384552-0.349652


In [7]:
# Use the updated Function 2 data
X_2 = updated_data[2]["X"]
y_2 = updated_data[2]["y"]

models_2 = {}

# Compare the original bounds with wider bounds
for name, upper_bound in [("A_original", 2.0), ("B_wider", 10.0)]:

    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(
            length_scale=[0.2] * X_2.shape[1],
            length_scale_bounds=(0.02, upper_bound),
            nu=2.5
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1)
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    model.fit(X_2, y_2)
    models_2[name] = model

    print(f"\nModel: {name}")
    print("Fitted kernel:", model.kernel_)
    print(
        "Log-marginal likelihood:",
        model.log_marginal_likelihood_value_
    )

c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(



Model: A_original
Fitted kernel: 0.926**2 * Matern(length_scale=[0.0672, 2], nu=2.5) + WhiteKernel(noise_level=0.00956)
Log-marginal likelihood: -10.673547667694812

Model: B_wider
Fitted kernel: 1.02**2 * Matern(length_scale=[0.0807, 10], nu=2.5) + WhiteKernel(noise_level=0.0132)
Log-marginal likelihood: -9.987270827354141


c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [8]:
# Generate the same candidate points for both models
rng_2 = np.random.default_rng(42)
candidates_2 = rng_2.uniform(0, 0.999999, size=(100000, 2))

# Round to the coordinates that would be submitted
candidates_2 = np.unique(np.round(candidates_2, 6), axis=0)

# Calculate distance to the nearest existing observation
nearest_distance_2 = np.min(
    np.linalg.norm(
        candidates_2[:, None, :] - X_2[None, :, :],
        axis=2
    ),
    axis=1
)

# Exclude points already evaluated
keep = nearest_distance_2 > 1e-6
candidates_2 = candidates_2[keep]
nearest_distance_2 = nearest_distance_2[keep]

# Compare UCB proposals
beta_2 = 2.0
proposals_2 = {}

for name, model in models_2.items():
    mean, std = model.predict(candidates_2, return_std=True)
    ucb = mean + beta_2 * std

    index = np.argmax(ucb)
    query = candidates_2[index]

    proposals_2[name] = {
        "query": query,
        "mean": float(mean[index]),
        "std": float(std[index]),
        "ucb": float(ucb[index])
    }

    print(f"\nModel: {name}")
    print("Proposed query:", "-".join(f"{x:.6f}" for x in query))
    print(f"Predicted output: {mean[index]:.6e}")
    print(f"Predictive standard deviation: {std[index]:.6e}")
    print(f"UCB score: {ucb[index]:.6e}")
    print(f"Distance to nearest observation: {nearest_distance_2[index]:.6f}")


Model: A_original
Proposed query: 0.963378-0.907944
Predicted output: 3.694443e-01
Predictive standard deviation: 2.082391e-01
UCB score: 7.859225e-01
Distance to nearest observation: 0.155074

Model: B_wider
Proposed query: 0.980795-0.613242
Predicted output: 3.852479e-01
Predictive standard deviation: 2.296621e-01
UCB score: 8.445720e-01
Distance to nearest observation: 0.167170


In [9]:
# Store the selected model and query for Function 2, round 2
selected_model_2 = models_2["B_wider"]
query_2_round2 = proposals_2["B_wider"]["query"].copy()

print("Function 2 — round 2:")
print("-".join(f"{x:.6f}" for x in query_2_round2))

Function 2 — round 2:
0.980795-0.613242


In [10]:
# Use the updated Function 3 data
X_3 = updated_data[3]["X"]
y_3 = updated_data[3]["y"]

models_3 = {}

# Compare the original bounds with wider bounds
for name, upper_bound in [("A_original", 2.0), ("B_wider", 10.0)]:

    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(
            length_scale=[0.2] * X_3.shape[1],
            length_scale_bounds=(0.02, upper_bound),
            nu=2.5
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1)
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    model.fit(X_3, y_3)
    models_3[name] = model

    print(f"\nModel: {name}")
    print("Fitted kernel:", model.kernel_)
    print(
        "Log-marginal likelihood:",
        model.log_marginal_likelihood_value_
    )

c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(



Model: A_original
Fitted kernel: 1.15**2 * Matern(length_scale=[2, 1.81, 0.0548], nu=2.5) + WhiteKernel(noise_level=1e-06)
Log-marginal likelihood: -18.174933001624773

Model: B_wider
Fitted kernel: 1.23**2 * Matern(length_scale=[10, 1.47, 0.0671], nu=2.5) + WhiteKernel(noise_level=1e-06)
Log-marginal likelihood: -17.69663969293194


c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [11]:
# Generate the same three-dimensional candidates for both models
rng_3 = np.random.default_rng(42)
candidates_3 = rng_3.uniform(0, 0.999999, size=(100000, 3))

# Round to portal-ready coordinates
candidates_3 = np.unique(np.round(candidates_3, 6), axis=0)

# Calculate distance to the nearest existing observation
nearest_distance_3 = np.min(
    np.linalg.norm(
        candidates_3[:, None, :] - X_3[None, :, :],
        axis=2
    ),
    axis=1
)

# Exclude points already evaluated
keep = nearest_distance_3 > 1e-6
candidates_3 = candidates_3[keep]
nearest_distance_3 = nearest_distance_3[keep]

# Compare UCB proposals
beta_3 = 2.0
proposals_3 = {}

for name, model in models_3.items():
    mean, std = model.predict(candidates_3, return_std=True)
    ucb = mean + beta_3 * std

    index = np.argmax(ucb)
    query = candidates_3[index]

    proposals_3[name] = {
        "query": query,
        "mean": float(mean[index]),
        "std": float(std[index]),
        "ucb": float(ucb[index])
    }

    print(f"\nModel: {name}")
    print("Proposed query:", "-".join(f"{x:.6f}" for x in query))
    print(f"Predicted output: {mean[index]:.6e}")
    print(f"Predictive standard deviation: {std[index]:.6e}")
    print(f"UCB score: {ucb[index]:.6e}")
    print(f"Distance to nearest observation: {nearest_distance_3[index]:.6f}")


Model: A_original
Proposed query: 0.986304-0.973945-0.396816
Predicted output: -5.980899e-02
Predictive standard deviation: 7.715154e-02
UCB score: 9.449409e-02
Distance to nearest observation: 0.205052

Model: B_wider
Proposed query: 0.880161-0.994823-0.405840
Predicted output: -4.922625e-02
Predictive standard deviation: 7.204119e-02
UCB score: 9.485614e-02
Distance to nearest observation: 0.226189


In [12]:
# Store the selected model and query for Function 3, round 2
selected_model_3 = models_3["B_wider"]
query_3_round2 = proposals_3["B_wider"]["query"].copy()

print("Function 3 — round 2:")
print("-".join(f"{x:.6f}" for x in query_3_round2))

Function 3 — round 2:
0.880161-0.994823-0.405840


In [13]:
# Use the updated Function 4 data
X_4 = updated_data[4]["X"]
y_4 = updated_data[4]["y"]

models_4 = {}

# Compare the original bounds with wider bounds
for name, upper_bound in [("A_original", 2.0), ("B_wider", 10.0)]:

    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(
            length_scale=[0.2] * X_4.shape[1],
            length_scale_bounds=(0.02, upper_bound),
            nu=2.5
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1)
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    model.fit(X_4, y_4)
    models_4[name] = model

    print(f"\nModel: {name}")
    print("Fitted kernel:", model.kernel_)
    print(
        "Log-marginal likelihood:",
        model.log_marginal_likelihood_value_
    )

c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-10. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(



Model: A_original
Fitted kernel: 3.21**2 * Matern(length_scale=[1.6, 1.45, 1.61, 1.45], nu=2.5) + WhiteKernel(noise_level=1e-10)
Log-marginal likelihood: -17.901558404320376

Model: B_wider
Fitted kernel: 3.21**2 * Matern(length_scale=[1.6, 1.45, 1.61, 1.45], nu=2.5) + WhiteKernel(noise_level=1e-10)
Log-marginal likelihood: -17.901558402746993


c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-10. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


In [14]:
# Keep the original bounds: both fits are practically identical
selected_model_4 = models_4["A_original"]

# Generate four-dimensional candidate points
rng_4 = np.random.default_rng(42)
candidates_4 = rng_4.uniform(0, 0.999999, size=(100000, 4))

# Round to portal-ready coordinates
candidates_4 = np.unique(np.round(candidates_4, 6), axis=0)

# Calculate distance to the nearest existing observation
nearest_distance_4 = np.min(
    np.linalg.norm(
        candidates_4[:, None, :] - X_4[None, :, :],
        axis=2
    ),
    axis=1
)

# Exclude points already evaluated
keep = nearest_distance_4 > 1e-6
candidates_4 = candidates_4[keep]
nearest_distance_4 = nearest_distance_4[keep]

# Predict outputs and uncertainty
mean_4, std_4 = selected_model_4.predict(
    candidates_4, return_std=True
)

# Select the candidate with the highest UCB score
beta_4 = 2.0
ucb_4 = mean_4 + beta_4 * std_4
index_4 = np.argmax(ucb_4)
query_4_round2 = candidates_4[index_4].copy()

print("Function 4 — round 2:")
print("-".join(f"{x:.6f}" for x in query_4_round2))
print(f"Predicted output: {mean_4[index_4]:.6e}")
print(f"Predictive standard deviation: {std_4[index_4]:.6e}")
print(f"UCB score: {ucb_4[index_4]:.6e}")
print(f"Distance to nearest observation: {nearest_distance_4[index_4]:.6f}")

Function 4 — round 2:
0.418847-0.410126-0.279614-0.437518
Predicted output: -1.236368e-01
Predictive standard deviation: 4.062609e-01
UCB score: 6.888850e-01
Distance to nearest observation: 0.079437


In [15]:
# Use the updated Function 5 data
X_5 = updated_data[5]["X"]
y_5 = updated_data[5]["y"]

models_5 = {}

# Compare the original bounds with wider bounds
for name, upper_bound in [("A_original", 2.0), ("B_wider", 10.0)]:

    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(
            length_scale=[0.2] * X_5.shape[1],
            length_scale_bounds=(0.02, upper_bound),
            nu=2.5
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1)
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    model.fit(X_5, y_5)
    models_5[name] = model

    print(f"\nModel: {name}")
    print("Fitted kernel:", model.kernel_)
    print(
        "Log-marginal likelihood:",
        model.log_marginal_likelihood_value_
    )

c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(



Model: A_original
Fitted kernel: 1.27**2 * Matern(length_scale=[2, 2, 0.842, 0.235], nu=2.5) + WhiteKernel(noise_level=6.36e-10)
Log-marginal likelihood: -17.195577549454917

Model: B_wider
Fitted kernel: 2.15**2 * Matern(length_scale=[10, 10, 1.28, 0.423], nu=2.5) + WhiteKernel(noise_level=1.26e-10)
Log-marginal likelihood: -10.290694951027543


c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [16]:
# Generate the same four-dimensional candidates for both models
rng_5 = np.random.default_rng(42)
candidates_5 = rng_5.uniform(0, 0.999999, size=(100000, 4))

# Round to portal-ready coordinates
candidates_5 = np.unique(np.round(candidates_5, 6), axis=0)

# Calculate distance to the nearest existing observation
nearest_distance_5 = np.min(
    np.linalg.norm(
        candidates_5[:, None, :] - X_5[None, :, :],
        axis=2
    ),
    axis=1
)

# Exclude points already evaluated
keep = nearest_distance_5 > 1e-6
candidates_5 = candidates_5[keep]
nearest_distance_5 = nearest_distance_5[keep]

# Compare UCB proposals
beta_5 = 2.0
proposals_5 = {}

for name, model in models_5.items():
    mean, std = model.predict(candidates_5, return_std=True)
    ucb = mean + beta_5 * std

    index = np.argmax(ucb)
    query = candidates_5[index]

    proposals_5[name] = {
        "query": query,
        "mean": float(mean[index]),
        "std": float(std[index]),
        "ucb": float(ucb[index])
    }

    print(f"\nModel: {name}")
    print("Proposed query:", "-".join(f"{x:.6f}" for x in query))
    print(f"Predicted output: {mean[index]:.6e}")
    print(f"Predictive standard deviation: {std[index]:.6e}")
    print(f"UCB score: {ucb[index]:.6e}")
    print(f"Distance to nearest observation: {nearest_distance_5[index]:.6f}")


Model: A_original
Proposed query: 0.165197-0.148071-0.999972-0.999842
Predicted output: 2.405524e+03
Predictive standard deviation: 2.906627e+02
UCB score: 2.986849e+03
Distance to nearest observation: 0.680231

Model: B_wider
Proposed query: 0.165197-0.148071-0.999972-0.999842
Predicted output: 2.662430e+03
Predictive standard deviation: 1.022829e+02
UCB score: 2.866996e+03
Distance to nearest observation: 0.680231


In [17]:
# Store the selected model and query for Function 5, round 2
selected_model_5 = models_5["B_wider"]
query_5_round2 = proposals_5["B_wider"]["query"].copy()

print("Function 5 — round 2:")
print("-".join(f"{x:.6f}" for x in query_5_round2))

Function 5 — round 2:
0.165197-0.148071-0.999972-0.999842


In [18]:
# Use the updated Function 6 data
X_6 = updated_data[6]["X"]
y_6 = updated_data[6]["y"]

models_6 = {}

# Compare the original bounds with wider bounds
for name, upper_bound in [("A_original", 2.0), ("B_wider", 10.0)]:

    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(
            length_scale=[0.2] * X_6.shape[1],
            length_scale_bounds=(0.02, upper_bound),
            nu=2.5
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1)
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    model.fit(X_6, y_6)
    models_6[name] = model

    print(f"\nModel: {name}")
    print("Fitted kernel:", model.kernel_)
    print(
        "Log-marginal likelihood:",
        model.log_marginal_likelihood_value_
    )


Model: A_original
Fitted kernel: 1.35**2 * Matern(length_scale=[0.475, 1.02, 0.918, 1.01, 0.972], nu=2.5) + WhiteKernel(noise_level=0.0253)
Log-marginal likelihood: -24.74157913527879

Model: B_wider
Fitted kernel: 1.35**2 * Matern(length_scale=[0.475, 1.02, 0.918, 1.01, 0.972], nu=2.5) + WhiteKernel(noise_level=0.0253)
Log-marginal likelihood: -24.741579136628612


In [19]:
# Keep the original bounds: both fits are practically identical
selected_model_6 = models_6["A_original"]

# Generate five-dimensional candidate points
rng_6 = np.random.default_rng(42)
candidates_6 = rng_6.uniform(0, 0.999999, size=(100000, 5))

# Round to portal-ready coordinates
candidates_6 = np.unique(np.round(candidates_6, 6), axis=0)

# Calculate distance to the nearest existing observation
nearest_distance_6 = np.min(
    np.linalg.norm(
        candidates_6[:, None, :] - X_6[None, :, :],
        axis=2
    ),
    axis=1
)

# Exclude points already evaluated
keep = nearest_distance_6 > 1e-6
candidates_6 = candidates_6[keep]
nearest_distance_6 = nearest_distance_6[keep]

# Predict outputs and uncertainty
mean_6, std_6 = selected_model_6.predict(
    candidates_6, return_std=True
)

# Select the candidate with the highest UCB score
beta_6 = 2.0
ucb_6 = mean_6 + beta_6 * std_6
index_6 = np.argmax(ucb_6)
query_6_round2 = candidates_6[index_6].copy()

print("Function 6 — round 2:")
print("-".join(f"{x:.6f}" for x in query_6_round2))
print(f"Predicted output: {mean_6[index_6]:.6e}")
print(f"Predictive standard deviation: {std_6[index_6]:.6e}")
print(f"UCB score: {ucb_6[index_6]:.6e}")
print(f"Distance to nearest observation: {nearest_distance_6[index_6]:.6f}")

Function 6 — round 2:
0.449640-0.366227-0.332069-0.902394-0.004736
Predicted output: -6.045831e-01
Predictive standard deviation: 2.754621e-01
UCB score: -5.365883e-02
Distance to nearest observation: 0.392694


In [21]:
# Use the updated Function 7 data
X_7 = updated_data[7]["X"]
y_7 = updated_data[7]["y"]

models_7 = {}

# Compare the original bounds with wider bounds
for name, upper_bound in [("A_original", 2.0), ("B_wider", 10.0)]:

    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(
            length_scale=[0.2] * X_7.shape[1],
            length_scale_bounds=(0.02, upper_bound),
            nu=2.5
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1)
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    model.fit(X_7, y_7)
    models_7[name] = model

    print(f"\nModel: {name}")
    print("Fitted kernel:", model.kernel_)
    print(
        "Log-marginal likelihood:",
        model.log_marginal_likelihood_value_
    )

c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(



Model: A_original
Fitted kernel: 1.11**2 * Matern(length_scale=[1.04, 0.591, 2, 0.774, 0.329, 0.594], nu=2.5) + WhiteKernel(noise_level=1.11e-09)
Log-marginal likelihood: -32.97038579168036

Model: B_wider
Fitted kernel: 1.25**2 * Matern(length_scale=[1.17, 0.617, 10, 0.91, 0.376, 0.709], nu=2.5) + WhiteKernel(noise_level=2.17e-09)
Log-marginal likelihood: -31.953422735976872


c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [22]:
# Keep the wider bounds: they provide a better fit
selected_model_7 = models_7["B_wider"]

# Generate six-dimensional candidate points
rng_7 = np.random.default_rng(42)
candidates_7 = rng_7.uniform(0, 0.999999, size=(100000, 6))

# Round to portal-ready coordinates
candidates_7 = np.unique(np.round(candidates_7, 6), axis=0)

# Calculate distance to the nearest existing observation
nearest_distance_7 = np.min(
    np.linalg.norm(
        candidates_7[:, None, :] - X_7[None, :, :],
        axis=2
    ),
    axis=1
)

# Exclude points already evaluated
keep = nearest_distance_7 > 1e-6
candidates_7 = candidates_7[keep]
nearest_distance_7 = nearest_distance_7[keep]

# Predict outputs and uncertainty
mean_7, std_7 = selected_model_7.predict(
    candidates_7, return_std=True
)

# Select the candidate with the highest UCB score
beta_7 = 2.0
ucb_7 = mean_7 + beta_7 * std_7
index_7 = np.argmax(ucb_7)
query_7_round2 = candidates_7[index_7].copy()

print("Function 7 — round 2:")
print("-".join(f"{x:.6f}" for x in query_7_round2))
print(f"Predicted output: {mean_7[index_7]:.6e}")
print(f"Predictive standard deviation: {std_7[index_7]:.6e}")
print(f"UCB score: {ucb_7[index_7]:.6e}")
print(f"Distance to nearest observation: {nearest_distance_7[index_7]:.6f}")

Function 7 — round 2:
0.002168-0.139823-0.026777-0.000298-0.281078-0.966260
Predicted output: 2.095893e+00
Predictive standard deviation: 2.393196e-01
UCB score: 2.574532e+00
Distance to nearest observation: 0.545723


In [23]:
# Store the selected model and query for Function 7, round 2
selected_model_7_round2 = selected_model_7
query_7_round2 = query_7_round2.copy()

print("Stored Function 7 model: B_wider")
print(
    "Stored Function 7 round-2 query:",
    "-".join(f"{x:.6f}" for x in query_7_round2)
)

Stored Function 7 model: B_wider
Stored Function 7 round-2 query: 0.002168-0.139823-0.026777-0.000298-0.281078-0.966260


In [24]:
# Use the updated Function 8 data
X_8 = updated_data[8]["X"]
y_8 = updated_data[8]["y"]

models_8 = {}

# Compare the original bounds with wider bounds
for name, upper_bound in [("A_original", 2.0), ("B_wider", 10.0)]:

    kernel = (
        ConstantKernel(1.0, (1e-3, 1e3))
        * Matern(
            length_scale=[0.2] * X_8.shape[1],
            length_scale_bounds=(0.02, upper_bound),
            nu=2.5
        )
        + WhiteKernel(
            noise_level=1e-6,
            noise_level_bounds=(1e-10, 1e-1)
        )
    )

    model = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=10,
        random_state=42
    )

    model.fit(X_8, y_8)
    models_8[name] = model

    print(f"\nModel: {name}")
    print("Fitted kernel:", model.kernel_)
    print(
        "Log-marginal likelihood:",
        model.log_marginal_likelihood_value_
    )

c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-pack


Model: A_original
Fitted kernel: 1.06**2 * Matern(length_scale=[1.18, 2, 0.914, 2, 2, 2, 1.24, 2], nu=2.5) + WhiteKernel(noise_level=1.62e-10)
Log-marginal likelihood: -16.59287641603569

Model: B_wider
Fitted kernel: 3.36**2 * Matern(length_scale=[3.05, 4.18, 2.14, 9.85, 10, 4.15, 3.03, 10], nu=2.5) + WhiteKernel(noise_level=1e-10)
Log-marginal likelihood: -2.513508684273667


c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 7 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\AG\BeatrizGIT\Master\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-10. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


In [25]:
# Generate eight-dimensional candidate points
rng_8 = np.random.default_rng(42)
candidates_8 = rng_8.uniform(0, 0.999999, size=(100000, 8))

# Round to portal-ready coordinates
candidates_8 = np.unique(np.round(candidates_8, 6), axis=0)

# Calculate distance to the nearest existing observation
nearest_distance_8 = np.min(
    np.linalg.norm(
        candidates_8[:, None, :] - X_8[None, :, :],
        axis=2
    ),
    axis=1
)

# Exclude points already evaluated
keep = nearest_distance_8 > 1e-6
candidates_8 = candidates_8[keep]
nearest_distance_8 = nearest_distance_8[keep]

# Compare the UCB proposals from both fitted models
beta_8 = 2.0
proposals_8 = {}

for name, model in models_8.items():

    mean_8, std_8 = model.predict(
        candidates_8,
        return_std=True
    )

    ucb_8 = mean_8 + beta_8 * std_8
    index_8 = np.argmax(ucb_8)

    proposals_8[name] = {
        "query": candidates_8[index_8].copy(),
        "mean": mean_8[index_8],
        "std": std_8[index_8],
        "ucb": ucb_8[index_8],
        "distance": nearest_distance_8[index_8]
    }

    print(f"\nModel: {name}")
    print(
        "Proposed query:",
        "-".join(
            f"{x:.6f}"
            for x in proposals_8[name]["query"]
        )
    )
    print(
        f"Predicted output: "
        f"{proposals_8[name]['mean']:.6e}"
    )
    print(
        f"Predictive standard deviation: "
        f"{proposals_8[name]['std']:.6e}"
    )
    print(
        f"UCB score: "
        f"{proposals_8[name]['ucb']:.6e}"
    )
    print(
        f"Distance to nearest observation: "
        f"{proposals_8[name]['distance']:.6f}"
    )


Model: A_original
Proposed query: 0.192843-0.059258-0.236797-0.023526-0.938709-0.700607-0.069105-0.152433
Predicted output: 9.801481e+00
Predictive standard deviation: 3.251710e-01
UCB score: 1.045182e+01
Distance to nearest observation: 0.723610

Model: B_wider
Proposed query: 0.192843-0.059258-0.236797-0.023526-0.938709-0.700607-0.069105-0.152433
Predicted output: 1.002433e+01
Predictive standard deviation: 1.886061e-01
UCB score: 1.040154e+01
Distance to nearest observation: 0.723610


In [26]:
# Keep the wider bounds: they provide a substantially better fit,
# while both models propose exactly the same candidate
selected_model_8 = models_8["B_wider"]

# Store the selected query for Function 8, round 2
query_8_round2 = proposals_8["B_wider"]["query"].copy()

print("Function 8 — round 2:")
print("-".join(f"{x:.6f}" for x in query_8_round2))
print(f"Predicted output: {proposals_8['B_wider']['mean']:.6e}")
print(
    f"Predictive standard deviation: "
    f"{proposals_8['B_wider']['std']:.6e}"
)
print(f"UCB score: {proposals_8['B_wider']['ucb']:.6e}")
print(
    f"Distance to nearest observation: "
    f"{proposals_8['B_wider']['distance']:.6f}"
)

Function 8 — round 2:
0.192843-0.059258-0.236797-0.023526-0.938709-0.700607-0.069105-0.152433
Predicted output: 1.002433e+01
Predictive standard deviation: 1.886061e-01
UCB score: 1.040154e+01
Distance to nearest observation: 0.723610


In [27]:
# Store the selected model and query for Function 8, round 2
selected_model_8_round2 = selected_model_8
query_8_round2 = query_8_round2.copy()

print("Stored Function 8 model: B_wider")
print(
    "Stored Function 8 round-2 query:",
    "-".join(f"{x:.6f}" for x in query_8_round2)
)

Stored Function 8 model: B_wider
Stored Function 8 round-2 query: 0.192843-0.059258-0.236797-0.023526-0.938709-0.700607-0.069105-0.152433


In [28]:
# ============================================================
# Final Round 2 queries
# ============================================================

round2_queries = {
    1: query_1_round2,
    2: query_2_round2,
    3: query_3_round2,
    4: query_4_round2,
    5: query_5_round2,
    6: query_6_round2,
    7: query_7_round2,
    8: query_8_round2,
}

for function_number, query in round2_queries.items():
    print(
        f"Function {function_number}: "
        + "-".join(f"{x:.6f}" for x in query)
    )

Function 1: 0.384552-0.349652
Function 2: 0.980795-0.613242
Function 3: 0.880161-0.994823-0.405840
Function 4: 0.418847-0.410126-0.279614-0.437518
Function 5: 0.165197-0.148071-0.999972-0.999842
Function 6: 0.449640-0.366227-0.332069-0.902394-0.004736
Function 7: 0.002168-0.139823-0.026777-0.000298-0.281078-0.966260
Function 8: 0.192843-0.059258-0.236797-0.023526-0.938709-0.700607-0.069105-0.152433


In [29]:
# Validate Round 2 query dimensions

expected_dimensions = {
    1: 2,
    2: 2,
    3: 3,
    4: 4,
    5: 4,
    6: 5,
    7: 6,
    8: 8,
}

for function_number, query in round2_queries.items():
    actual_dim = len(query)
    expected_dim = expected_dimensions[function_number]

    status = "OK" if actual_dim == expected_dim else "ERROR"

    print(
        f"Function {function_number}: "
        f"{actual_dim} dimensions "
        f"(expected {expected_dim}) — {status}"
    )

Function 1: 2 dimensions (expected 2) — OK
Function 2: 2 dimensions (expected 2) — OK
Function 3: 3 dimensions (expected 3) — OK
Function 4: 4 dimensions (expected 4) — OK
Function 5: 4 dimensions (expected 4) — OK
Function 6: 5 dimensions (expected 5) — OK
Function 7: 6 dimensions (expected 6) — OK
Function 8: 8 dimensions (expected 8) — OK
